In [ ]:
import pandas as pd
import numpy as np
import time
from datetime import datetime
import os

In [ ]:
def get_adb_windows(df, window_size):
    column_time = ['unixTime']
    ADB_Times = []
    pd.options.display.float_format = '{:.0f}'.format # nearest whole number for unixTime
    columns_to_search = ['isAggressiveStreering', 'isHardAcc', 'isHardBrak','isSpeeding']
    for column in columns_to_search:
        true_values = df[df[column] == True]
        if not true_values.empty:
#             print(f"True values found in {column}:")
#             print(true_values[column_time])

            ADB_Times += true_values['unixTime'].tolist()
#         else:
#             print(f"No true values found in {column}.")
                
    # Convert Unix timestamps to datetime objects and store in a new list
    ADB_datetimes = [datetime.fromtimestamp(t) for t in ADB_Times]

    # Initialize an empty list to store the filtered windows
    adb_windows = []

    # FINDING THE 5 MINUTE INTERVALS
    for adb_time in ADB_Times:    
        # Calculate the start and end times for the 5-minute window preceding the ADB event
        end_time = pd.to_datetime(adb_time, unit='s')
        start_time = end_time - window_size

        # Extract the rows within the 5-minute window
        window_adb = df[(df['unixTime'] >= start_time.timestamp()) & (df['unixTime'] <= end_time.timestamp())]

        # Append the filtered window to the list of filtered windows
        adb_windows.append(window_adb)

    # Check if adb_windows is empty, if so return None or an empty DataFrame
    if not adb_windows:
        return 0, []
    # Combine the filtered windows into a single DataFrame
    adb_df = pd.concat(adb_windows)
    no_adb_windows = len(adb_windows)
    print(f"Number of adb windows: {no_adb_windows}")
    return no_adb_windows, ADB_Times

In [ ]:
def non_adb_intervals(df, removal_window, ADB_Times, window_size=pd.Timedelta(minutes=5), overlap_size=pd.Timedelta(minutes=2.5), pct_start_moving=0.8):
    # If there are no ADB cases, return the original DataFrame
    if len(ADB_Times) == 0:
        df_remaining = df 
    else:
        # REMOVE 30 MIN WINDOW BEFORE ADB
        adb_remove = []
        if len(ADB_Times) == 1:
            adb_time = ADB_Times[0]
            end_time = pd.to_datetime(adb_time, unit='s')
            start_time = end_time - window_size
            adb_remove = df[(df['unixTime'] >= start_time.timestamp()) & (df['unixTime'] <= end_time.timestamp())]
        else:
            for adb_time in ADB_Times:    
                end_time = pd.to_datetime(adb_time, unit='s')
                start_time = end_time - window_size
                window_remove = df[(df['unixTime'] >= start_time.timestamp()) & (df['unixTime'] <= end_time.timestamp())]
                adb_remove.append(window_remove)
            adb_remove = pd.concat(adb_remove).reset_index(drop=True)

        mask = df.isin(adb_remove)
        df_remaining = df[~mask.any(axis=1)]
        
    # REMOVE <% START MOVING
    overlap_seconds = overlap_size.total_seconds()
    windows = []
    i = 0
    while i < len(df_remaining):
        start_time = df_remaining.iloc[i]['recordTime']
        end_time = start_time + window_size
        window = df_remaining[(df_remaining['recordTime'] >= start_time) & (df_remaining['recordTime'] < end_time)]
        windows.append(window)
        i += int(window_size.total_seconds() - overlap_seconds)

    # Filter the windows to only include windows with no ADB events and intervals where start moving is true for more than the specified percentage of the interval
    filtered_windows = []
    for window in windows:
        # Calculate the percentage of time the driver is moving
        pct_start_moving_window = window['isStartMoving'].sum() / len(window)

        # Check if the window meets the criteria (over specified percentage moving time, no ADB events)
        if pct_start_moving_window >= pct_start_moving:
            filtered_windows.append(window)

#     # Combine the filtered windows into a single DataFrame
#     result_df = pd.concat(filtered_windows).reset_index(drop=True)
    no_windows = len(filtered_windows)
    print(f"Number of non adb windows: {no_windows}")
    
#     # Return the result DataFrame
    return no_windows

In [ ]:
date_file_path = "C:/Users/wzqwa/OneDrive - Imperial College London/Imperial Year 4/FYP/Data example/Provided DB_Final.xlsx"
Dates = pd.read_excel(date_file_path, sheet_name="Trip_sum")["Date"].tolist()
dates = Dates[:-13]
dates = [date.strftime('%Y-%m-%d') for date in dates]
print(dates)

In [ ]:
file_names = []
dummyf = []

window_size = pd.Timedelta(minutes=5)
overlap_size = pd.Timedelta(minutes=2.5)
removal_window = pd.Timedelta(minutes=30)
pct_start_moving = 0.5    

for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".xlsx"):
            file_names.append(file)
            dummyf.append(root)

total_no_windows = 0
total_adb_windows = 0
for root, filename in zip(dummyf, file_names):
    file_name_no_ext = os.path.splitext(filename)[0]
    if file_name_no_ext in dates:
        file_path = os.path.join(root, filename)
        df = pd.read_excel(file_path)
        print(file_name_no_ext)
        # Convert recordTime column to datetime format
        df['recordTime'] = pd.to_datetime(df['recordTime'], format='%Y%m%d%H%M%S')
        # Convert datetime format to Unix timestamps
        df['unixTime'] = (df['recordTime'] - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s')
    #     df = df[df["token"].isin(tokens)]
#         print("Included token:", df["token"].unique())
        no_adb_windows, ADB_Times = get_adb_windows(df, window_size)
        no_windows = non_adb_intervals(df, removal_window, ADB_Times, window_size, overlap_size, pct_start_moving)
        total_adb_windows = total_adb_windows + no_adb_windows
        total_no_windows = total_no_windows + no_windows
        print(f"Total number of adb windows: {total_adb_windows}")
        print(f"Total number of non adb windows: {total_no_windows}")